In [ ]:
from websocket import create_connection
import json
import pandas as pd
from datetime import date
from time import sleep

In [ ]:
## web socket url
socket = 'wss://data.tradingview.com/socket.io/websocket'
# change session id based on current session value
session_id = 'cs_UPhNJp9xQbTR'
## example messages to be sent
# ~m~55~m~{"m":"chart_create_session","p":["cs_yDp6P0icM81I",""]}
# ~m~117~m~{"m":"resolve_symbol","p":["cs_yDp6P0icM81I","sds_sym_1","={\"adjustment\":\"splits\",\"symbol\":\"NSE:HDFCBANK\"}"]}
# ~m~82~m~{"m":"create_series","p":["cs_yDp6P0icM81I","sds_1","s1","sds_sym_1","1W",300,""]}

downloads = './data/nifty500_5yr_daily/'

In [ ]:
# create message as per template and send
def create_msg(ws, fun, arg):
    ms = json.dumps({"m":fun,"p":arg})
    msg = f'~m~{len(ms)}~m~{ms}'
    ws.send(msg)

# format result data into pandas dataframe
def format_data(data):
    start = data.find('"s":[')
    end = data.find(',"ns":')
    trimmed = data[start+4 : end]
    final_data = [x["v"] for x in json.loads(data[start+4 : end])]
    # print(final_data)
    return final_data

In [ ]:
def fetch_historic_data(symbol):
    # resolve_symbol argument
    sess_msg = [session_id,""]    
    resolve_symbol = '={\"adjustment\":\"splits\",\"symbol\":\"NSE:' + symbol + '\"}'
    res_sym_msg = [session_id,"sds_sym_1",resolve_symbol]
    ser_msg = [session_id,"sds_1","s1","sds_sym_1","1D",1000,""]
    # create fresh websocket connection
    ws = create_connection(socket)
    # send messages to websocket
    create_msg(ws, 'chart_create_session', sess_msg)
    create_msg(ws, 'resolve_symbol', res_sym_msg)
    create_msg(ws, 'create_series', ser_msg)
    
    while True :
        # receive response for the messages sent
        res = ws.recv()
        if "series_completed" in res:
            formatted_data = format_data(res)
            # convert json to pandas dataframe for analysis
            df_result = pd.DataFrame(formatted_data, columns=['date', 'open', 'high', 'low', 'close', 'volume'])
            # convert epoch timestamp to date yyyy-mm-dd format
            df_result['date'] = df_result['date'].apply(lambda x : str(date.fromtimestamp(x)))
            df_result.sort_values('date', inplace=True)
            # calculate average
            df_result['avg'] = df_result.apply(lambda row : int((row['open'] + row['high'] + row['low'] + row['close'])/4), axis=1)
            # shutdown websocket connection for next iteration to work properly
            ws.shutdown()
            # break the infinite while loop
            break
    
    return df_result

In [ ]:
# run for a sample symbol
df_result = fetch_historic_data('BAJAJ_AUTO')
df_result

In [ ]:
# if loop breaks before completing all symbols, add the completed symbols to this list
completed_list = []

# use index/list of your choice
nifty_500 = pd.read_csv('./data/downloads/ind_nifty500list.csv')
nifty_500

In [ ]:
symbols = list(sorted(set(nifty_500['Symbol']) - set(completed_list)))

for i, symbol in enumerate(symbols) :
    if '-' in symbol:
        symbol = symbol.replace('-', '_')
    df_result = fetch_historic_data(symbol)
    df_result.to_csv(downloads + symbol + '.csv', index=None)
    completed_list.append(symbol)
    sleep(1)
    print(i+1, symbol)